In [12]:
from langgraph.graph import StateGraph, END
from typing import TypedDict
from langchain_openai import ChatOpenAI
from typing import Annotated
from langgraph.graph.message import add_messages
from dotenv import load_dotenv
from langgraph.checkpoint.memory import MemorySaver
from langchain_core.messages import HumanMessage
import os

In [5]:
load_dotenv()
llm = ChatOpenAI(model="gpt-4o-mini")

In [6]:
class MessagesState(TypedDict):
    messages: Annotated[list, add_messages]

In [7]:
def ask_llm(state: MessagesState) -> MessagesState:
    response = llm.invoke(state["messages"])
    return {"messages": [response]}

In [8]:
graph = StateGraph(MessagesState)
graph.add_node("ask_llm", ask_llm)
graph.set_entry_point("ask_llm")
graph.add_edge("ask_llm", END)
memory = MemorySaver()
app = graph.compile()

In [13]:
def send_message(thread_id: str, user_text: str) -> str:
    """Send a message and return the AI reply. This writes to the DB."""
    result = app.invoke(
        {"messages": [HumanMessage(content=user_text)]}
    )
    return result["messages"][-1].content

In [15]:
result  = app.invoke({"messages": [HumanMessage(content="Hello, how are you?")]})
print(result["messages"][-1].content)

Hello! I'm just a program, but I'm here and ready to help you. How can I assist you today?
